# CLIPSeg rd64-refined — DIMER text-prompted image segmentation tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/clipseg-segmentation-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/clipseg-segmentation-pipeline/blob/main/tutorials/clipseg_segmentation_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-CIDAS%2Fclipseg--rd64--refined-ffcc4d?style=flat)](https://huggingface.co/CIDAS/clipseg-rd64-refined) [![Upstream](https://img.shields.io/badge/Upstream-timojl%2Fclipseg-181717?style=flat&logo=github&logoColor=white)](https://github.com/timojl/clipseg) [![arXiv](https://img.shields.io/badge/arXiv-2112.10003-b31b1b.svg)](https://arxiv.org/abs/2112.10003)

**Profile:** `TASK-INFERENCE`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** zero-shot (text-prompted) image segmentation — one image plus 1–16 free-text phrases → one binary mask and probability map per phrase — using the pinned `CIDAS/clipseg-rd64-refined` weights

**This notebook is standalone.** It carries the repository's pipeline module (`src/clipseg_segmentation_pipeline/pipeline.py` at revision `230938f2675a`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `999e0328d9e10b484360c477313983f9afdd7050` (~605 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned snapshot, obtains the tutorial sample automatically, validates it into an input manifest before the model runs, runs the task locally in this kernel, writes the evaluation report, and exports machine-readable outputs with provenance. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5).

**Bring Your Own Data:** After the sample workflow completes, set `USE_BYOD = True` in the sample cell and re-run from that cell to supply your own input. It passes through the same notebook-local validation, task, evaluation-report and export cells as the sample; the expected input format, the ceilings and the privacy guidance are stated in the Prerequisites and in the sample cell, and the upload stays inside this runtime. BYOD is optional and never part of the default path.

At inference the CLIPSeg model (a frozen CLIP ViT-B/16 image encoder and CLIP text encoder joined by a small transformer decoder with 64-dimensional reduced activations and a refined transposed convolution head; about 151M parameters, the decoder trained on PhraseCut phrase–mask pairs) conditions the decoder on each text phrase and produces one 352×352 logit map per phrase; the carried module passes the logits through a sigmoid, resamples the probability map to the input size, and thresholds it into a binary mask under a caller-owned `threshold`. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting happens in this notebook — the upstream checkpoint supplies the weights, processor and tokenizer, and the carried module adds snapshot verification, the input contract (image side ceilings, 1–16 distinct phrases up to 64 characters, a threshold in [0, 1]), a fixed output contract (mask, probability map, area fraction, tight box and maximum probability per phrase), and the `mask_iou`, `validate_inputs` and `evaluation_report` helpers. The default sample is a flat scene of coloured shapes drawn in code with the exact masks they were drawn from, so the per-phrase `mask_iou` and their mean are demonstration (plumbing) evidence for one drawing, not a PhraseCut benchmark.

**Learning objectives:** install the pinned runtime, read what the carried pipeline module guarantees, resolve and digest-verify the immutable upstream model revision, draw a synthetic scene with known masks (or upload your own photograph and type your own phrases) and validate it into an input manifest, choose a threshold, run the supported task, read the masks correctly (an uncalibrated sigmoid per pixel, independent per phrase, no class exclusivity), exercise an optional BYOD path, produce an evaluation report that is `sample-sanity` with `mask_iou` and `miou` only when reference masks exist and `not-measurable` otherwise, and export the masks, an overlay and provenance.

**This notebook does not demonstrate:** Instance separation (one mask per phrase, even when several objects match it), panoptic or semantic labelling of every pixel (masks are independent per phrase and may overlap or leave pixels unassigned), the image-prompt (one-shot) conditioning mode the upstream model also supports, high-resolution boundary accuracy (the decoder works at 352×352 and the mask is resampled), batch throughput, evaluation on PhraseCut, Pascal-5i or COCO (not bundled; only drawn shapes are scored here), and any training. The model was trained on photographs with phrase annotations; flat drawings, documents, medical and satellite imagery, and non-English phrases are outside what this notebook measures, and a confident-looking mask carries no signal.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU and uses CUDA automatically when available; inference is float32 on both. CPU is adequate: the repository's model card records 5.0 s to load and 0.8 s for six phrases on a 640×480 drawn scene (0.15 s for one) in the Windows venv (Intel Core Ultra 9 275HX). The pinned `torch==2.14.0` install and the 603 MB checkpoint are the large downloads of the run.
- **Knowledge:** basic Python, NumPy and PIL; what a per-pixel sigmoid is and why thresholding it is a decision the caller owns; what intersection-over-union of masks measures and why a few drawn shapes are not a benchmark.
- **Data:** the default sample is a deterministic 640×480 scene drawn in code with Pillow (a red circle, a blue square, a yellow triangle and a green ground band on an off-white background; no text rendering, so its digest is stable across Pillow builds) with the exact boolean masks the shapes were drawn from, so nothing is downloaded and no private data is needed. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction. Expected BYOD input: one image decodable by Pillow (PNG/JPEG/WebP and similar), any colour mode, sides between 16 and 4096 px, plus your own phrases typed into the form field; no reference masks exist for uploads, so their report is `not-measurable`. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `CIDAS/clipseg-rd64-refined` snapshot (~605 MB in total) at revision `999e0328d9e1…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'transformers==4.57.6',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'huggingface-hub==0.36.2',
]
NOTEBOOK_SOURCE = {
    'repository': 'clipseg-segmentation-pipeline',
    'repository_revision': '230938f2675a8577db6f22d4518567f2ef52a96f',
    'embedded_module': 'src/clipseg_segmentation_pipeline/pipeline.py',
    'embedded_modules': ['src/clipseg_segmentation_pipeline/pipeline.py'],
    'module_sha256': 'dea04612b6f1e2fa75e1a8994289ffe0b90709878d4c87b84f30650a121e4dba',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/clipseg_segmentation_pipeline/` @ `230938f2675a`)

The next 1 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/1:** `src/clipseg_segmentation_pipeline/pipeline.py`

In [ ]:
"""Text-prompted (zero-shot) image segmentation with the pinned ``CIDAS/clipseg-rd64-refined`` checkpoint.

The class loads the processor and model only from a digest-verified local snapshot (``weights/<key>/``)
or, when explicitly allowed, from the Hugging Face Hub at the pinned revision — always with
``trust_remote_code=False``: the CLIPSeg architecture comes from the pinned ``transformers`` release,
the weights are SafeTensors, and no model-repository code is executed.
"""

from __future__ import annotations

import hashlib
import json
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
from PIL import Image

MODEL_ID = "CIDAS/clipseg-rd64-refined"
MODEL_REVISION = "999e0328d9e10b484360c477313983f9afdd7050"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "clipseg-rd64-refined"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# Threshold on the per-pixel sigmoid of the decoder logits. 0.5 is the natural cut of a sigmoid and
# the value the smoke run used; the sigmoid is not calibrated, so the deployment owns tuning it on its
# own labelled masks.
MASK_THRESHOLD = 0.5
# Decoder output resolution: logits are 352x352 for every input (preprocessor_config.json resizes to
# 352x352 without preserving aspect ratio); the pipeline resamples the probability map back to the
# input size bilinearly.
LOGIT_SIZE = 352
# Input ceilings. Image cost is bounded by the fixed resize; the side ceiling only guards memory during
# decoding and resampling. Each prompt is one CLIP text query (77-token context); phrases are short.
MAX_IMAGE_SIDE = 4096
MIN_IMAGE_SIDE = 16
MAX_PROMPTS = 16
MAX_PROMPT_CHARS = 64
MAX_TEXT_TOKENS = 77


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
    }


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def mask_iou(a: np.ndarray, b: np.ndarray) -> float:
    """Intersection-over-union of two boolean masks of the same shape; the building block for any mIoU."""
    a_bool, b_bool = np.asarray(a, dtype=bool), np.asarray(b, dtype=bool)
    if a_bool.shape != b_bool.shape:
        raise ValueError(f"mask shapes differ: {a_bool.shape} vs {b_bool.shape}")
    union = np.logical_or(a_bool, b_bool).sum()
    return float(np.logical_and(a_bool, b_bool).sum() / union) if union else 0.0


def mask_bbox(mask: np.ndarray) -> list[int] | None:
    """Tight xyxy pixel box around the true pixels of a mask, or ``None`` for an empty mask."""
    rows = np.flatnonzero(np.asarray(mask, dtype=bool).any(axis=1))
    cols = np.flatnonzero(np.asarray(mask, dtype=bool).any(axis=0))
    if rows.size == 0 or cols.size == 0:
        return None
    return [int(cols[0]), int(rows[0]), int(cols[-1]) + 1, int(rows[-1]) + 1]


def format_prompts(prompts: Sequence[str]) -> list[str]:
    """Validate a list of phrases and normalise them: stripped, whitespace-collapsed, lower-cased,
    trailing full stop removed, distinct. The pipeline passes the caller's phrases through otherwise
    unchanged (the upstream examples use plain noun phrases such as "a cat")."""
    if isinstance(prompts, str) or not isinstance(prompts, Sequence):
        raise TypeError("prompts must be a list of phrases, not a single string")
    if not 1 <= len(prompts) <= MAX_PROMPTS:
        raise ValueError(f"prompt count {len(prompts)} outside 1..MAX_PROMPTS {MAX_PROMPTS}")
    cleaned: list[str] = []
    for phrase in prompts:
        if not isinstance(phrase, str):
            raise TypeError(f"prompt must be str, got {type(phrase).__name__}")
        text = " ".join(phrase.split()).strip().rstrip(".").strip().lower()
        if not text:
            raise ValueError("prompt phrases must not be empty")
        if len(text) > MAX_PROMPT_CHARS:
            raise ValueError(
                f"prompt {text[:12]!r}... is {len(text)} chars > MAX_PROMPT_CHARS {MAX_PROMPT_CHARS}"
            )
        cleaned.append(text)
    if len(set(cleaned)) != len(cleaned):
        raise ValueError("prompt phrases must be distinct after normalisation")
    return cleaned


def validate_image(image: Any) -> Image.Image:
    if not isinstance(image, Image.Image):
        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")
    width, height = image.size
    if min(width, height) < MIN_IMAGE_SIDE:
        raise ValueError(f"image side {min(width, height)} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")
    if max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"image side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")
    return image.convert("RGB")


def _check_threshold(name: str, value: Any) -> float:
    if isinstance(value, bool) or not isinstance(value, int | float) or not 0.0 <= value <= 1.0:
        raise ValueError(f"{name} must be a number in [0, 1], got {value!r}")
    return float(value)


INPUT_SCHEMA: dict[str, Any] = {
    "input": "one PIL.Image.Image (any mode, converted to RGB) plus 1..MAX_PROMPTS free-text phrases",
    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "prompts": [1, MAX_PROMPTS],
    "prompt_chars": [1, MAX_PROMPT_CHARS],
    "prompt_tokens_per_query": [1, MAX_TEXT_TOKENS],
    "threshold": [0.0, 1.0],
    "preprocessing": (
        f"image converted to RGB and resized to {LOGIT_SIZE}x{LOGIT_SIZE} (aspect ratio not preserved, "
        "ImageNet mean/std); phrases normalised into one CLIP text query each (format_prompts); the "
        f"decoder's {LOGIT_SIZE}x{LOGIT_SIZE} logits are passed through a sigmoid and resampled "
        "bilinearly to the input size; the mask is probability >= threshold"
    ),
    "output": (
        "per prompt: a boolean mask and a float32 probability map at input resolution, the mask's area "
        "fraction, tight box and maximum probability; probabilities are uncalibrated sigmoids"
    ),
}


def _check_inputs(image: Any, prompts: Any, threshold: Any) -> tuple[Image.Image, list[str], float]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the checked request.

    ``segment`` and ``validate_inputs`` both route through this function so their acceptance
    criteria cannot diverge.
    """
    rgb = validate_image(image)
    queries = format_prompts(prompts)
    checked = _check_threshold("threshold", threshold)
    return rgb, queries, checked


def validate_inputs(
    image: Image.Image,
    prompts: Sequence[str],
    *,
    threshold: float = MASK_THRESHOLD,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict).

    Rejection is reported by raising exactly as ``segment`` would; a caller that wants the finding
    recorded catches the exception and stores ``str(exc)`` under ``findings``.
    """
    _rgb, queries, checked = _check_inputs(image, prompts, threshold)
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry (segment takes one image)")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [
            {
                "id": names[0] if names else "image-0",
                "mode": image.mode,
                "size": list(image.size),
                "n_prompts": len(prompts),
            }
        ],
        "queries": queries,
        "threshold": checked,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any],
    reference_masks: Mapping[str, np.ndarray] | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``reference_masks`` (phrase -> boolean mask at input resolution) the report carries one
    ``mask_iou`` entry per reference and their mean (``miou``) as sample-sanity evidence; without them
    the verdict is ``not-measurable`` and the report says what labelled data would make the task
    measurable.
    """
    segments = list(result["segments"])
    by_prompt = {segment["prompt"]: segment for segment in segments}
    base = {
        "task": "zero-shot (text-prompted) binary segmentation, one mask per phrase",
        "decision_rule": (
            "each phrase yields a per-pixel sigmoid over the decoder logits; a pixel belongs to the mask "
            "when that sigmoid reaches the threshold; the sigmoid is uncalibrated and masks of different "
            "phrases are independent (they may overlap or leave pixels unassigned)"
        ),
        "threshold": result.get("threshold", MASK_THRESHOLD),
        "sample_kind": sample_kind,
        "n_prompts": len(segments),
        "area_fractions": {segment["prompt"]: segment["area_fraction"] for segment in segments},
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if not reference_masks:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no reference masks were supplied for the evaluated image",
            "needs": (
                "labelled masks on your own images with a phrase vocabulary matching the prompts, scored "
                "per phrase with mask_iou and aggregated into mean IoU at a stated threshold; no such "
                "labelled set ships with this repository"
            ),
        }
    metrics = []
    for phrase, reference in reference_masks.items():
        key = format_prompts([phrase])[0]
        if key not in by_prompt:
            raise ValueError(f"reference phrase {phrase!r} was not among the segmented prompts")
        metrics.append(
            {
                "id": "mask_iou",
                "reference": key,
                "value": mask_iou(by_prompt[key]["mask"], reference),
                "reference_area_fraction": float(np.asarray(reference, dtype=bool).mean()),
                "predicted_area_fraction": by_prompt[key]["area_fraction"],
                "estimation": "one reference mask per phrase on a single scene, no dispersion estimate",
            }
        )
    miou = sum(entry["value"] for entry in metrics) / len(metrics)
    metrics.append(
        {
            "id": "miou",
            "value": miou,
            "estimation": f"mean of {len(metrics)} mask_iou value(s) on one scene, no dispersion estimate",
        }
    )
    return {
        **base,
        "metrics": metrics,
        "verdict": "sample-sanity",
        "reason": (
            f"{len(metrics) - 1} reference mask(s) on one tutorial sample; geometry sanity evidence, "
            "not a segmentation benchmark"
        ),
        "needs": (
            "a labelled mask set from the deployment domain with a matching phrase vocabulary for any "
            "mean-IoU claim"
        ),
    }


@dataclass
class ClipSegSegmentationPipeline:
    """Text-prompted (zero-shot) binary segmentation over the pinned CLIPSeg rd64-refined checkpoint."""

    _runner: Callable[[Image.Image, list[str]], np.ndarray]
    device: str

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> ClipSegSegmentationPipeline:
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            source, kwargs = str(root), {"local_files_only": True}
        elif allow_download:
            source, kwargs = MODEL_ID, {}
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage {MODEL_ID}@{MODEL_REVISION} under weights/{MODEL_KEY}"
            )
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import CLIPSegForImageSegmentation, CLIPSegProcessor

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        processor = CLIPSegProcessor.from_pretrained(
            source, revision=MODEL_REVISION, trust_remote_code=False, **kwargs
        )
        model = CLIPSegForImageSegmentation.from_pretrained(
            source, revision=MODEL_REVISION, trust_remote_code=False, dtype=torch.float32, **kwargs
        )
        model = model.to(resolved_device).eval()

        def runner(image: Image.Image, queries: list[str]) -> np.ndarray:
            # One image copy per phrase: CLIPSeg conditions the decoder on each text query separately.
            inputs = processor(
                text=queries, images=[image] * len(queries), padding=True, return_tensors="pt"
            ).to(resolved_device)
            with torch.inference_mode():
                logits = model(**inputs).logits
            if logits.dim() == 2:  # a single prompt may come back squeezed
                logits = logits.unsqueeze(0)
            probs = torch.sigmoid(logits).unsqueeze(1)
            probs = torch.nn.functional.interpolate(
                probs, size=(image.height, image.width), mode="bilinear", align_corners=False
            )
            return probs.squeeze(1).float().cpu().numpy()

        return cls(runner, resolved_device)

    def segment(
        self,
        image: Image.Image,
        prompts: Sequence[str],
        *,
        threshold: float = MASK_THRESHOLD,
    ) -> dict[str, Any]:
        """Segment each phrase in ``prompts``; masks and probability maps are at input resolution."""
        rgb, queries, checked = _check_inputs(image, prompts, threshold)
        probs = np.asarray(self._runner(rgb, queries), dtype=np.float32)
        if probs.shape != (len(queries), rgb.height, rgb.width):
            raise RuntimeError(
                f"backend returned probability maps of shape {probs.shape}, "
                f"expected {(len(queries), rgb.height, rgb.width)}"
            )
        if probs.min() < 0.0 or probs.max() > 1.0:
            raise RuntimeError("backend returned probabilities outside [0, 1]")
        segments = []
        for query, prob in zip(queries, probs, strict=True):
            mask = prob >= checked
            segments.append(
                {
                    "prompt": query,
                    "mask": mask,
                    "probability": prob,
                    "area_fraction": float(mask.mean()),
                    "max_probability": float(prob.max()),
                    "bbox": mask_bbox(mask),
                }
            )
        return {
            "segments": segments,
            "queries": queries,
            "threshold": checked,
            "width": rgb.width,
            "height": rgb.height,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `8`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `999e0328d9e1…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `ClipSegSegmentationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "clipseg-rd64-refined",
  "modelId": "CIDAS/clipseg-rd64-refined",
  "revision": "999e0328d9e10b484360c477313983f9afdd7050",
  "files": [
    {
      "path": "README.md",
      "bytes": 596,
      "sha256": "c1e26ac022542015a04d804460bdc1159d753c4349b628e086f127475d2fcf25"
    },
    {
      "path": "config.json",
      "bytes": 4732,
      "sha256": "c023375966d31b3b1392764f7bd91df47098ce19f62f11b0263d8eedcf708bcd"
    },
    {
      "path": "merges.txt",
      "bytes": 524619,
      "sha256": "9fd691f7c8039210e0fced15865466c65820d09b63988b0174bfe25de299051a"
    },
    {
      "path": "model.safetensors",
      "bytes": 603049496,
      "sha256": "d00ca85d6b859f9d07b7cfb8ef26fe9771cb275b34c9368f2ecf603139307f55"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 380,
      "sha256": "4fb09ebcfd7651205ca8299b993c30088e5535ef350a72d46c5c4580eeac0440"
    },
    {
      "path": "special_tokens_map.json",
      "bytes": 472,
      "sha256": "c4864a9376a8401918425bed71fc14fc0e81f9b59ec45c1cf96cccb2df508eac"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 974,
      "sha256": "4c75d57fd9bd0be8478ad2d6f8b9cebdd4a45338eb108547329c2b6333476ca6"
    },
    {
      "path": "vocab.json",
      "bytes": 1059962,
      "sha256": "e089ad92ba36837a0d31433e555c8f45fe601ab5c221d4f607ded32d9f7a4349"
    }
  ],
  "totalBytes": 604641231
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = ClipSegSegmentationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Draw the synthetic scene or optional BYOD

The default sample is **synthetic** and carries its own references: a red circle, a blue square and a yellow triangle on an off-white background above a green ground band are drawn with Pillow at 640×480, and the same drawing calls produce the boolean reference mask for each shape — the same scene the repository's smoke run used. Four phrases name the four regions and two more (`a cat`, `the sky`) name things that are not there, so the notebook shows both a mask that should be found and one that should stay empty. The reference masks are the references for the `mask_iou` sanity check later. They are not a labelled dataset, so nothing here is a PhraseCut measurement. The image digest is printed for the record. BYOD is optional and disabled by default; when enabled, upload one image and type your phrases (one per line) — no reference masks exist for them, so the evaluation report will be `not-measurable`.

The mask threshold is a **caller-owned request parameter**: `threshold` cuts the per-pixel sigmoid (`MASK_THRESHOLD = 0.5` is the package default, the natural cut of a sigmoid, not a calibration — the smoke run recorded IoU 0.81–0.96 at 0.3, 0.91–0.96 at 0.5 and 0.83–0.90 at 0.7 on this scene). Nothing is validated in this cell — the next section hands the image and the phrases to the pipeline's own validation stage, which is the only checker. Look for a dictionary naming the sample kind, the image size and digest, the threshold and the phrases.

In [ ]:
import hashlib
import io

import numpy as np
from PIL import Image, ImageDraw

USE_BYOD = False  # @param {type:"boolean"}
byod_prompts = 'a person\na dog'  # @param {type:"string"}
threshold = 0.5  # @param {type:"number"}


def synthetic_scene(width=640, height=480):
    """Coloured shapes drawn with Pillow (no text); returns image + {phrase: boolean reference mask}."""
    image = Image.new('RGB', (width, height), (245, 245, 240))
    d = ImageDraw.Draw(image)
    shapes = [
        ('green grass', 'rectangle', [0, 320, 640, 480], (60, 179, 75)),
        ('a red circle', 'ellipse', [80, 80, 260, 260], (220, 40, 40)),
        ('a blue square', 'rectangle', [340, 90, 560, 300], (40, 70, 200)),
        ('a yellow triangle', 'polygon', [(200, 460), (320, 330), (440, 460)], (250, 200, 30)),
    ]
    masks = {}
    for phrase, kind, geometry, colour in shapes:
        getattr(d, kind)(geometry, fill=colour)
        reference = Image.new('1', image.size)
        getattr(ImageDraw.Draw(reference), kind)(geometry, fill=1)
        masks[phrase] = np.array(reference, dtype=bool)
    return image, masks


if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    image_name = next(iter(uploaded))
    image = Image.open(io.BytesIO(uploaded[image_name]))
    image.load()
    prompts = [line.strip() for line in byod_prompts.splitlines() if line.strip()]
    reference_masks = None
    sample_kind = 'BYOD'
else:
    # Deterministic drawing: no randomness and no text rendering, so no seed is needed and the digest is stable.
    image, reference_masks = synthetic_scene()
    prompts = list(reference_masks) + ['a cat', 'the sky']  # two absent phrases on purpose
    image_name = 'synthetic_shapes_640x480.png'
    sample_kind = 'synthetic'

image_sha256 = hashlib.sha256(np.asarray(image.convert('RGB')).tobytes()).hexdigest()
print({'sample_kind': sample_kind, 'name': image_name, 'mode': image.mode, 'size': image.size, 'rgb_sha256': image_sha256, 'threshold': threshold, 'prompts': prompts, 'has_reference_masks': reference_masks is not None})

## 5. Validate the request → input manifest

`validate_inputs` is the pipeline's public validation stage: it applies exactly the checks `segment` applies — image type and sides `MIN_IMAGE_SIDE`..`MAX_IMAGE_SIDE` px, 1..`MAX_PROMPTS` distinct non-empty phrases of at most `MAX_PROMPT_CHARS` characters (normalised by `format_prompts`), and a threshold in `[0, 1]` — and returns an **input manifest** naming the schema (including the 352×352 resize that does not preserve aspect ratio, the sigmoid and the resampling), the input's observed mode and size, the normalised phrases, the threshold and the verdict. The manifest is written to `outputs/clipseg_segmentation_input_manifest.json`. To show what rejection looks like, the cell also validates a duplicated phrase and records the pipeline's own error message as a finding. Inside the pipeline the image is converted to RGB and resized; nothing else is dropped or altered. The pipeline cannot tell whether a phrase names anything in the image: that contract is the caller's, and an absent phrase simply yields an empty (or spurious) mask.

In [ ]:
import json
import os

os.makedirs('outputs', exist_ok=True)
print({'ceilings': {'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'LOGIT_SIZE': LOGIT_SIZE, 'MAX_PROMPTS': MAX_PROMPTS, 'MAX_PROMPT_CHARS': MAX_PROMPT_CHARS, 'MAX_TEXT_TOKENS': MAX_TEXT_TOKENS, 'MASK_THRESHOLD': MASK_THRESHOLD}})
input_manifest = validate_inputs(image, prompts, threshold=threshold, names=[image_name])
# Demonstrate rejection on a request that breaks the contract; the finding is recorded, not swallowed.
try:
    validate_inputs(image, ['a red circle', 'A red circle.'])
except ValueError as exc:
    input_manifest['findings'].append({'input': 'duplicate-phrase-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/clipseg_segmentation_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps(input_manifest, indent=2))

## 6. Segment the phrases and read the output correctly

`segment` returns one entry per phrase with `mask` (a boolean array at input resolution), `probability` (the float32 sigmoid map it was cut from), `area_fraction`, `max_probability`, a tight `bbox` around the mask (or `None` when it is empty), plus the normalised `queries`, the `threshold`, the image size and the model identity. **The probabilities are uncalibrated sigmoids**: a 0.9 pixel is not 90 % likely to belong to the phrase, and the masks of different phrases are independent — they can overlap, and pixels can belong to none. Greedy thresholding is deterministic on a fixed device and dtype; CUDA kernels can shift probabilities slightly, so GPU and CPU masks need not match at the boundary. Every phrase costs one decoder pass over the same encoded image (about 0.15 s each on the reference CPU). As recorded in the model card, the repository's CPU smoke on this same scene found all four shapes with IoU 0.91–0.96 at 0.5, left `a cat` and `the sky` empty (maximum probability ≤ 0.01), and on a blank white image or uniform noise left `a red circle` empty too (maximum 0.02) — an absent phrase usually yields an empty mask here, but that is an observation, not a guarantee.

In [ ]:
import time

t0 = time.time()
result = pipe.segment(image, prompts, threshold=threshold)
elapsed = round(time.time() - t0, 2)
print({'device': pipe.device, 'seconds': elapsed, 'n_prompts': len(result['queries']), 'threshold': result['threshold']})
for segment in result['segments']:
    print(f"{segment['prompt']:20s} area={segment['area_fraction']:.3f}  max_p={segment['max_probability']:.2f}  bbox={segment['bbox']}")

## 7. Evaluate → evaluation report

`evaluation_report` is the pipeline's public evaluation stage and always produces a report. No accuracy is reported by default: segmentation quality needs labelled masks on images from the deployment domain with a matching phrase vocabulary, and this repository ships none (PhraseCut is not bundled). The repository's metric helper is `mask_iou` — intersection-over-union of two boolean masks — and when reference masks are supplied the report carries one `mask_iou` entry per phrase (with the reference and predicted area fractions) and their mean `miou`, with the verdict `sample-sanity`. On the synthetic path those references are shapes **you drew yourself**, so a high IoU proves only that the input contract, resize, decoder, sigmoid, resampling and thresholding round-trip. On BYOD no reference masks exist, the verdict is `not-measurable`, and the report states what would make the task measurable. The report is written to `outputs/clipseg_segmentation_evaluation_report.json`.

In [ ]:
report = evaluation_report(result, reference_masks, sample_kind=sample_kind)
with open('outputs/clipseg_segmentation_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(json.dumps({k: v for k, v in report.items() if k != 'metrics'}, indent=2))
for metric in report['metrics']:
    print(f"{metric['id']:10} {metric['value']:.3f}  {metric.get('reference', '')}  ({metric['estimation']})")
if report['verdict'] == 'not-measurable':
    print('No reference masks exist for these phrases, so nothing is scored; inspect the overlay yourself.')

## 8. Export outputs and provenance

Machine-readable JSON preserves the per-phrase statistics (area fraction, maximum probability, tight box), the queries and threshold, the evaluation report, the input manifest, the sample identity and digest, the notebook's source (repository, revision, embedded module digest, generator), the model identifier, the immutable model revision, the model licence, and the runtime identity (Python, `torch`, `transformers`, device); the masks themselves are written as one 8-bit PNG per phrase (0/255) and the probability maps as one `.npz`, because arrays do not belong in JSON. An overlay PNG tints each phrase's mask in its own colour on the image for visual inspection (a supplement to, not a replacement for, the machine-readable files). No credentials are recorded.

In [ ]:
import re

palette = [(220, 40, 40), (40, 70, 200), (250, 200, 30), (60, 179, 75), (160, 60, 200), (0, 170, 170)]
base = np.asarray(image.convert('RGB'), dtype=np.float32)
overlay = base.copy()
for index, segment in enumerate(result['segments']):
    colour = np.array(palette[index % len(palette)], dtype=np.float32)
    overlay[segment['mask']] = 0.45 * overlay[segment['mask']] + 0.55 * colour
Image.fromarray(overlay.round().astype(np.uint8)).save('outputs/clipseg_segmentation_overlay.png')
mask_files = {}
for segment in result['segments']:
    slug = re.sub(r'[^a-z0-9]+', '_', segment['prompt']).strip('_')
    path = f'outputs/clipseg_segmentation_mask_{slug}.png'
    Image.fromarray((segment['mask'].astype(np.uint8) * 255)).save(path)
    mask_files[segment['prompt']] = path
np.savez_compressed('outputs/clipseg_segmentation_probabilities.npz', **{segment['prompt']: segment['probability'] for segment in result['segments']})
payload = {
    'segments': [{k: v for k, v in segment.items() if k not in ('mask', 'probability')} for segment in result['segments']],
    'queries': result['queries'],
    'threshold': result['threshold'],
    'mask_files': mask_files,
    'evaluation_report': report,
    'input_manifest': input_manifest,
    'sample': {'kind': sample_kind, 'name': image_name, 'size': list(image.size), 'rgb_sha256': image_sha256, 'prompts': prompts, 'has_reference_masks': reference_masks is not None},
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'device': pipe.device,
    },
}
with open('outputs/clipseg_segmentation_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The masks are per-pixel sigmoid cuts of a decoder conditioned on your phrase; nothing in the output scores a mask as a whole, the probabilities are uncalibrated, and masks of different phrases are independent. On the drawn scene the `mask_iou` values in the evaluation report compare the masks with shapes you drew yourself and the verdict is `sample-sanity`, which proves only that the input contract, resize, decoder, sigmoid, resampling and thresholding work (the repository's smoke run scored IoU 0.91–0.96 on the four shapes and left the two absent phrases empty); they say nothing about photographs, cluttered scenes, thin or small objects (the decoder works at 352×352), phrases naming attributes or relations, or non-English prompts, and a BYOD result is a single-image observation with the verdict `not-measurable`. **An absent phrase is not guaranteed an empty mask** and a present one is not guaranteed a full one: the threshold trades area for precision (0.3 grew every mask, 0.7 shrank them in the smoke run), so choose it on your own labelled masks. The pipeline provides no instance separation, no exhaustive pixel labelling, no image-prompt mode, no benchmark evaluation and no training capability.

Successful execution proves that the recorded repository revision's pipeline module, carried in this notebook, can acquire and digest-verify the pinned model, validate the demonstrated request, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime — without the repository being reachable. It does **not** establish benchmark superiority, deployment calibration, safety for high-consequence decisions, or production fitness on an unseen domain.

**Next experiments:** set `threshold` to 0.3 and 0.7 and watch the IoUs move; rename `a blue square` to `a blue rectangle` or `a blue box`; ask for `a shape` and see which pixels the decoder assigns; enable `USE_BYOD` with a photograph you know, then build your own boolean reference masks and pass them to `evaluation_report` to see the verdict switch to `sample-sanity`.

## References

- Repository README: https://github.com/kurtvalcorza/clipseg-segmentation-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/clipseg-segmentation-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/clipseg-segmentation-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/CIDAS/clipseg-rd64-refined
- Upstream code: https://github.com/timojl/clipseg
- Image Segmentation Using Text and Image Prompts (Lüddecke and Ecker, 2021): https://arxiv.org/abs/2112.10003
- PhraseCut: Language-based Image Segmentation in the Wild (Wu et al., 2020): https://arxiv.org/abs/2008.01187